# 04 — Customer RFM Analysis
This notebook calculates Recency, Frequency, and Monetary (RFM) metrics for customer segmentation.

**Concepts demonstrated:** groupby, custom aggregation, date difference, quantile bucketing

In [ ]:
import pandas as pd
from pathlib import Path

## 1. Load and prepare data

In [ ]:
RAW = Path('../data/raw')
orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW / 'order_items.csv')

df = orders.merge(order_items, on='order_id', how='inner')
df['revenue'] = df['quantity'] * df['item_price']
df = df[df['order_status'].isin(['paid', 'shipped', 'delivered'])].copy()
df.head()

## 2. Calculate RFM metrics

In [ ]:
snapshot_date = pd.Timestamp('2024-12-31')
rfm = df.groupby('customer_id').agg(
    recency_days=('order_date', lambda s: (snapshot_date - s.max()).days),
    frequency_orders=('order_id', 'nunique'),
    monetary_spend=('revenue', 'sum')
).reset_index()
rfm.head()

## 3. Create quartile-based RFM scores

In [ ]:
rfm['r_rank'] = pd.qcut(rfm['recency_days'], 4, labels=[4,3,2,1]).astype(int)
rfm['f_rank'] = pd.qcut(rfm['frequency_orders'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['m_rank'] = pd.qcut(rfm['monetary_spend'], 4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['r_rank']*100 + rfm['f_rank']*10 + rfm['m_rank']
rfm.sort_values('rfm_score', ascending=False).head(10)

## 4. Optional customer segment labels

In [ ]:
def label_segment(score):
    if score >= 444:
        return 'VIP'
    elif score >= 333:
        return 'High Value'
    elif score >= 222:
        return 'Medium'
    return 'Low'

rfm['segment'] = rfm['rfm_score'].apply(label_segment)
rfm['segment'].value_counts()

## 5. Export for Tableau / Power BI

In [ ]:
OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)
rfm.to_csv(OUT / 'customer_rfm_analysis.csv', index=False)
print('Saved:', OUT / 'customer_rfm_analysis.csv')

## Try it yourself

### Questions
1. Find the top 10 VIP customers.
2. Compare average spend by segment.
3. Add a chart later in Power BI or Tableau using this output.

### Answers

In [ ]:
# 1. Top 10 VIP customers
vip_customers = rfm[rfm['segment'] == 'VIP'].sort_values('monetary_spend', ascending=False).head(10)
display(vip_customers)

# 2. Average spend by segment
segment_spend = rfm.groupby('segment', as_index=False)['monetary_spend'].mean().rename(columns={'monetary_spend': 'avg_spend'})
display(segment_spend)

# 3. Chart-ready output already exists in customer_rfm_analysis.csv
print('Use customer_rfm_analysis.csv in Power BI or Tableau for segment charts.')